In [ ]:
import os
from copy import deepcopy
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
import torchvision
from torchvision.datasets import STL10
from torchvision import transforms
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
DATASET_PATH = "../SimClr/data"
CHECKPOINT_PATH = "./saved_models/tutorial17"
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
%load_ext tensorboard
log_dir = "./saved_models/tutorial17"
os.makedirs(log_dir, exist_ok=True)
%tensorboard --logdir {log_dir}

data

In [ ]:
#返回两个经数据增强后的图像
class ContrastiveTransformations(object):

    def __init__(self, base_transforms, n_views=2):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transforms(x) for i in range(self.n_views)]
#augmentation的五种方式
contrast_transforms = transforms.Compose([transforms.RandomHorizontalFlip(0.5),
                                          
                                          transforms.RandomResizedCrop(size=96),
                                            #颜色调整概率0.8
                                          transforms.RandomApply([
                                              transforms.ColorJitter(
                                                    #限制亮度在[0.5-1.5]
                                                  brightness=0.5,
                                                  #对比度
                                                    contrast=0.5,
                                                    #饱和度
                                                    saturation=0.5,
                                                    #色相偏移
                                                    hue=0.1)
                                          ], p=0.8),
                                          #转为灰度图像
                                          transforms.RandomGrayscale(p=0.2),
                                          #高斯卷积核模糊处理
                                          transforms.RandomApply(
                                            [ transforms.GaussianBlur(
                                                    kernel_size=9,
                                                    sigma=(0.1, 2.0)
                                                ) ],p=0.5),
                                          #totensor会将数据从（0，255）转到（0，1）
                                          transforms.ToTensor(),
                                          transforms.Normalize((0.5,), (0.5,))
                                         
                                         ])
unlabeled_data = STL10(root=DATASET_PATH, split='unlabeled', download=True,
                       transform=ContrastiveTransformations(contrast_transforms, n_views=2))
# finetune阶段数据处理
finetune_train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        size=96,
        scale=(0.7, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])
# 测试阶段数据处理
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])
# 有标签 train 集：用于微调整个 BYOL backbone
finetune_train_data = STL10(
    root=DATASET_PATH,
    split="train",
    download=True,
    transform=finetune_train_transforms
)
# test 集：只用于最终评估
test_data = STL10(
    root=DATASET_PATH,
    split="test",
    download=True,
    transform=test_transforms
)

model

In [ ]:
class BYOL(pl.LightningModule):
    def __init__(
        self,
        hidden_dim,
        lr,
        weight_decay,
        max_epochs=500,
        momentum=0.996
    ):
        super().__init__()
        self.save_hyperparameters()

        assert 0 <= momentum < 1

        # online network：正常反向传播
        self.online_encoder = self._build_encoder(hidden_dim)

        # target network：仅通过 EMA 更新
        self.target_encoder = self._build_encoder(hidden_dim)
        self.target_encoder.load_state_dict(
            self.online_encoder.state_dict()
        )

        #教师模型不需要计算梯度，进行动量更新
        for param in self.target_encoder.parameters():
            param.requires_grad = False

        #predictor head
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim, bias=False),
            nn.BatchNorm1d(4 * hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(4 * hidden_dim, hidden_dim)
        )
    #构建resnet18网络，包括projection head
    def _build_encoder(self, hidden_dim):
        """ResNet18 backbone + projection head。"""

        encoder = torchvision.models.resnet18(
            weights=None,
            num_classes=4 * hidden_dim
        )

        # 适配 STL10 的 96×96 图像
        encoder.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False
        )
        encoder.maxpool = nn.Identity()

        # projector
        encoder.fc = nn.Sequential(
            encoder.fc,                       # 512 -> 4 * hidden_dim
            nn.BatchNorm1d(4 * hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(
                4 * hidden_dim,
                hidden_dim,
                bias=False
            ),
            nn.BatchNorm1d(
                hidden_dim,
                affine=False
            )
        )

        return encoder
    #动量更新target network
    @torch.no_grad()
    def _momentum_update_target_encoder(self):
        """target = m * target + (1-m) * online"""

        momentum = self.hparams.momentum

        for online_param, target_param in zip(
            self.online_encoder.parameters(),
            self.target_encoder.parameters()
        ):  #mul_表示原地乘法
            target_param.data.mul_(momentum)
            target_param.data.add_(
                online_param.data,
                alpha=1.0 - momentum
            )
    #计算byol的mse损失
    @staticmethod
    def regression_loss(prediction, target):
        """
        BYOL 的负余弦相似度损失：
        2 - 2 * cosine_similarity
        """
        prediction = F.normalize(prediction, dim=1)
        target = F.normalize(target.detach(), dim=1)

        return 2 - 2 * (prediction * target).sum(dim=1).mean()
    #计算样本与正样本的损失
    def byol_loss(self, batch, mode="train"):
        images, _ = batch
        view_1, view_2 = images

        # online 分支
        online_z1 = self.online_encoder(view_1)
        online_z2 = self.online_encoder(view_2)

        prediction_1 = self.predictor(online_z1)
        prediction_2 = self.predictor(online_z2)

        # target 分支不参与反向传播
        with torch.no_grad():
            if mode == "train":
                self._momentum_update_target_encoder()

            target_z1 = self.target_encoder(view_1)
            target_z2 = self.target_encoder(view_2)

        # 交叉预测：
        # view1 的 online 输出预测 view2 的 target 输出
        # view2 的 online 输出预测 view1 的 target 输出
        loss_1 = self.regression_loss(
            prediction_1,
            target_z2
        )
        loss_2 = self.regression_loss(
            prediction_2,
            target_z1
        )

        loss = 0.5 * (loss_1 + loss_2)
        if mode != 'test': 
            self.log(
                f"{mode}_loss",
                loss,
                on_step=False,
                on_epoch=True,
                prog_bar=True,
                batch_size=view_1.shape[0]
            )

        return loss
    def training_step(self, batch, batch_idx):
        return self.byol_loss(batch, mode="train")
    def validation_step(self, batch, batch_idx):
        return self.byol_loss(batch, mode="val")
    def configure_optimizers(self):
        # 只优化 online encoder 和 predictor
        trainable_parameters = (
            list(self.online_encoder.parameters()) +
            list(self.predictor.parameters())
        )

        optimizer = optim.AdamW(
            trainable_parameters,
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.lr / 50
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch"
            }
        }
class STL10Classifier(pl.LightningModule):
    def __init__(
        self,
        lr=1e-4,
        weight_decay=1e-4,
        max_epochs=50,
        num_classes=10,
        pretrained_byol=None
    ):
        super().__init__()

        self.save_hyperparameters(
            ignore=["pretrained_byol"]
        )

        if pretrained_byol is not None:
            # 复制 BYOL 的 online encoder
            self.model = deepcopy(
                pretrained_byol.online_encoder
            )

            # BYOL 的 fc 是 projector
            # fc[0] 是原来 ResNet18 的分类层
            feature_dim = self.model.fc[0].in_features

            # 删除 BYOL projector，换成分类层
            self.model.fc = nn.Linear(
                feature_dim,
                num_classes
            )

            print("使用 BYOL 预训练参数")

        else:
            # 随机初始化的 ResNet18 对照模型
            self.model = torchvision.models.resnet18(
                weights=None,
                num_classes=num_classes
            )

            # 与 BYOL backbone 保持完全相同的网络结构
            self.model.conv1 = nn.Conv2d(
                in_channels=3,
                out_channels=64,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            )

            self.model.maxpool = nn.Identity()

            print("使用随机初始化参数")

        # 明确设置：所有参数都参与反向传播
        for param in self.model.parameters():
            param.requires_grad = True

    def forward(self, images):
        return self.model(images)

    def _calculate_loss(self, batch, mode):
        images, labels = batch

        logits = self(images)
        loss = F.cross_entropy(logits, labels)
        #prediction的维度是[B,1]
        predictions = logits.argmax(dim=1)
        #计算准确率
        accuracy = (
            predictions == labels
        ).float().mean()
        if mode != 'test':
            self.log(
                f"{mode}_loss",
                loss,
                on_step=False,
                on_epoch=True,
                prog_bar=(mode != "train"),
                batch_size=images.shape[0]
            )
        if mode == 'test':
            self.log(
                f"{mode}_acc",
                accuracy,
                on_step=False,
                on_epoch=True,
                prog_bar=True,
                batch_size=images.shape[0]
            )

        return loss

    def training_step(self, batch, batch_idx):
        return self._calculate_loss(
            batch,
            mode="train"
        )

    def test_step(self, batch, batch_idx):
        self._calculate_loss(
            batch,
            mode="test"
        )

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.lr / 100
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch"
            }
        }

train

In [ ]:
#预训练函数
def train_byol(batch_size, max_epochs=50, **kwargs):
    checkpoint_callback = ModelCheckpoint(
        save_weights_only=True,
        save_last=True,
        save_top_k=0
        )
    tb_logger = TensorBoardLogger(
        save_dir=CHECKPOINT_PATH,
        name="BYOL",
        version="pretrain",
        default_hp_metric=False,
        log_graph=False
    )
    trainer = pl.Trainer(
        logger=tb_logger,
        default_root_dir=os.path.join(
            CHECKPOINT_PATH,
            "BYOL"
        ),
        accelerator=(
            "gpu" if torch.cuda.is_available()
            else "cpu"
        ),
        devices=1,
        precision=(
            "16-mixed" if torch.cuda.is_available()
            else "32-true"
        ),
        max_epochs=max_epochs,
        callbacks=[
            checkpoint_callback
        ]
    )

    train_loader = data.DataLoader(
        unlabeled_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
        num_workers=0
    )

    model = BYOL(
        max_epochs=max_epochs,
        **kwargs
    )

    trainer.fit(model, train_loader)

    return model
#微调或随机参数训练函数
def train_and_test_classifier(
    model,
    model_name,
    batch_size=64,
    max_epochs=50
):
    checkpoint_callback = ModelCheckpoint(
        save_weights_only=True,
        save_last=True,
        save_top_k=0
    )
    tb_logger = TensorBoardLogger(
        save_dir=CHECKPOINT_PATH,
        name=model_name,
        version="train",
        default_hp_metric=False,
        log_graph=False
    )
    train_loader = data.DataLoader(
        finetune_train_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = data.DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    trainer = pl.Trainer(
        logger=tb_logger,
        default_root_dir=os.path.join(
            CHECKPOINT_PATH,
            model_name
        ),
        accelerator=(
            "gpu" if torch.cuda.is_available()
            else "cpu"
        ),
        devices=1,
        precision=(
            "16-mixed" if torch.cuda.is_available()
            else "32-true"
        ),
        max_epochs=max_epochs,
        callbacks=[
            checkpoint_callback
        ],
        deterministic=True
    )

    # 只使用 train 数据训练
    trainer.fit(
        model,
        train_dataloaders=train_loader
    )

    # 训练结束后，只在这里使用 test 数据
    test_result = trainer.test(
        model,
        dataloaders=test_loader,
        verbose=True
    )

    result = {
        "model_name": model_name,
        "test_acc": test_result[0]["test_acc"]
    }

    return model, result
#预训练
byol_model = train_byol(
    batch_size=64,
    hidden_dim=128,
    lr=5e-4,
    weight_decay=1e-4,
    momentum=0.996,
    max_epochs=10
)
#微调
finetune_model = STL10Classifier(
    pretrained_byol=byol_model,
    lr=3e-4,
    weight_decay=1e-4,
    max_epochs=50
)
finetune_model, finetune_result = train_and_test_classifier(
    model=finetune_model,
    model_name="BYOL_Finetune",
    batch_size=64,
    max_epochs=50
)
#随机参数训练
random_model = STL10Classifier(
    pretrained_byol=None,
    lr=3e-4,
    weight_decay=1e-4,
    max_epochs=50
)
random_model, random_result = train_and_test_classifier(
    model=random_model,
    model_name="Random_ResNet18",
    batch_size=64,
    max_epochs=50
)